In [8]:
import pandas as pd
import random

# Config
NER_DATASET = r""
RECURRENCE_DATASET = r""
OUTPUT_CSV = ""
NUM_PACIENTES = 250
RATIO_POSITIVOS = 0.25
CONTEXT_CHARS = 30


df_ner = pd.read_csv(NER_DATASET, encoding='utf-8')
df_consultas = pd.read_csv(RECURRENCE_DATASET, encoding='utf-8')

print(f"Entidades NER totales: {len(df_ner)}")

df_progresion = df_ner[df_ner['label'] == 'PROGRESION'].copy()
print(f"Entidades PROGRESION: {len(df_progresion)}")
df_consultas['patient_id'] = df_consultas['id_paciente'] + '_' + df_consultas['id_consulta'] + '_' + df_consultas['fecha']
df_merged = df_progresion.merge(df_consultas, on='patient_id', how='inner')
print(f"Tras merge: {len(df_merged)}")
consultas_pos = df_merged[df_merged['RECURRENCIA'] == 1].copy()
consultas_neg = df_merged[df_merged['RECURRENCIA'] == 0].copy()

print(f"\nConsultas totales:")
print(f"  Con recurrencia: {len(consultas_pos)}")
print(f"  Sin recurrencia: {len(consultas_neg)}")
total_pos = len(consultas_pos)
num_neg = int(total_pos * (1 - RATIO_POSITIVOS) / RATIO_POSITIVOS)

print(f"\nSeleccion para balance .25:")
print(f"  Positivas: {total_pos} (todas)")
print(f"  Negativas: {num_neg} (de {len(consultas_neg)} disponibles)")
if len(consultas_neg) >= num_neg:
    consultas_neg_sel = consultas_neg.sample(n=num_neg, random_state=42)
else:
    print(f"  Advertencia: Solo hay {len(consultas_neg)} negativas, tomando todas")
    consultas_neg_sel = consultas_neg

df_final = pd.concat([consultas_pos, consultas_neg_sel], ignore_index=True)

print(f"\nDataset combinado:")
print(f"  Total consultas: {len(df_final)}")
print(f"  Con recurrencia: {(df_final['RECURRENCIA'] == 1).sum()} ({(df_final['RECURRENCIA'] == 1).sum() / len(df_final) * 100:.1f}%)")
print(f"  Sin recurrencia: {(df_final['RECURRENCIA'] == 0).sum()} ({(df_final['RECURRENCIA'] == 0).sum() / len(df_final) * 100:.1f}%)")

# Pacientes unicos
print(f"  Pacientes unicos: {df_final['id_paciente'].nunique()}")

# Ampliar span
def ampliar(row):
    texto = row['TEXTO_LIMPIO']
    start = int(row['start'])
    end = int(row['end'])
    
    nuevo_start = max(0, start - CONTEXT_CHARS)
    nuevo_end = min(len(texto), end + CONTEXT_CHARS)
    
    return texto[nuevo_start:nuevo_end]

df_final['span_ampliado'] = df_final.apply(ampliar, axis=1)

resultado = df_final[[
    'id_paciente', 
    'id_consulta', 
    'fecha', 
    'RECURRENCIA', 
    'span_ampliado',
    'TEXTO_LIMPIO',
    'score'
]].copy()

resultado.columns = [
    'id_paciente', 
    'id_consulta', 
    'fecha_consulta', 
    'recurrencia', 
    'span_ampliado',
    'texto_limpio',
    'score_ner'
]

resultado.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nGuardado: {OUTPUT_CSV}")

Entidades NER totales: 380967
Entidades PROGRESION: 11479
Tras merge: 11479

Consultas totales:
  Con recurrencia: 382
  Sin recurrencia: 11097

Seleccion para balance .25:
  Positivas: 382 (todas)
  Negativas: 1146 (de 11097 disponibles)

Dataset combinado:
  Total consultas: 1528
  Con recurrencia: 382 (25.0%)
  Sin recurrencia: 1146 (75.0%)
  Pacientes unicos: 396

Guardado: spans_context_250_balanced.csv


In [9]:
INPUT_CSV = ""
OUTPUT_CSV = ""
MIN_SCORE = 0.75

df = pd.read_csv(INPUT_CSV, encoding='utf-8')

print(f"Filas iniciales: {len(df)}")
print(f"Consultas unicas: {df['id_consulta'].nunique()}")
print(f"Spans unicos: {df['span_ampliado'].nunique()}")

print(f"\nFiltrando por score {MIN_SCORE}...")
df_filtered = df[df['score_ner'] >= MIN_SCORE].copy()

print(f"Tras filtrar score: {len(df_filtered)} filas")
print(f"  Eliminadas: {len(df) - len(df_filtered)}")
print("\nEliminando spans duplicados")
df_no_dup_span = df_filtered.sort_values('score_ner', ascending=False).drop_duplicates(
    subset=['span_ampliado'], 
    keep='first'
)

print(f"Tras eliminar spans duplicados: {len(df_no_dup_span)} filas")
print(f"  Eliminadas: {len(df_filtered) - len(df_no_dup_span)}")

print("\nEliminando consultas duplicadas")
df_final = df_no_dup_span.sort_values('score_ner', ascending=False).drop_duplicates(
    subset=['id_consulta'], 
    keep='first'
)

print(f"Tras eliminar consultas duplicadas: {len(df_final)} filas")
print(f"  Eliminadas: {len(df_no_dup_span) - len(df_final)}")

# Estadisticas finales
print(f"\nDataset final:")
print(f"  Filas: {len(df_final)}")
print(f"  Pacientes unicos: {df_final['id_paciente'].nunique()}")
print(f"  Consultas unicas: {df_final['id_consulta'].nunique()}")
print(f"  Con recurrencia: {(df_final['recurrencia'] == 1).sum()} ({(df_final['recurrencia'] == 1).sum() / len(df_final) * 100:.1f}%)")
print(f"  Sin recurrencia: {(df_final['recurrencia'] == 0).sum()} ({(df_final['recurrencia'] == 0).sum() / len(df_final) * 100:.1f}%)")
print(f"  Score medio: {df_final['score_ner'].mean():.3f}")
print(f"  Score minimo: {df_final['score_ner'].min():.3f}")

# Guardar
df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nGuardado: {OUTPUT_CSV}")

Filas iniciales: 1528
Consultas unicas: 1304
Spans unicos: 1321

Filtrando por score 0.75...
Tras filtrar score: 1322 filas
  Eliminadas: 206

Eliminando spans duplicados
Tras eliminar spans duplicados: 1131 filas
  Eliminadas: 191

Eliminando consultas duplicadas
Tras eliminar consultas duplicadas: 998 filas
  Eliminadas: 133

Dataset final:
  Filas: 998
  Pacientes unicos: 376
  Consultas unicas: 998
  Con recurrencia: 235 (23.5%)
  Sin recurrencia: 763 (76.5%)
  Score medio: 0.956
  Score minimo: 0.753

Guardado: spans_cleaned.csv


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Config
INPUT_CSV = ""
OUTPUT_TRAIN = "train.csv"
OUTPUT_VAL = "validation.csv"
OUTPUT_TEST = "test.csv"

TRAIN_SIZE = 0.70  
VAL_SIZE = 0.15    
TEST_SIZE = 0.15   

RANDOM_STATE = 42
df = pd.read_csv(INPUT_CSV, encoding='utf-8')

print(f"Total filas: {len(df)}")
print(f"Con recurrencia: {(df['recurrencia'] == 1).sum()}")
print(f"Sin recurrencia: {(df['recurrencia'] == 0).sum()}")

# Particionar manteniendo proporcion de recurrencia (stratify)
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_SIZE + TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=df['recurrencia']
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=temp_df['recurrencia']
)

# Estadisticas
print(f"\nTrain ({len(train_df)} filas, {len(train_df)/len(df)*100:.1f}%):")
print(f"  Con recurrencia: {(train_df['recurrencia'] == 1).sum()} ({(train_df['recurrencia'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"  Sin recurrencia: {(train_df['recurrencia'] == 0).sum()} ({(train_df['recurrencia'] == 0).sum()/len(train_df)*100:.1f}%)")

print(f"\nValidation ({len(val_df)} filas, {len(val_df)/len(df)*100:.1f}%):")
print(f"  Con recurrencia: {(val_df['recurrencia'] == 1).sum()} ({(val_df['recurrencia'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"  Sin recurrencia: {(val_df['recurrencia'] == 0).sum()} ({(val_df['recurrencia'] == 0).sum()/len(val_df)*100:.1f}%)")

print(f"\nTest ({len(test_df)} filas, {len(test_df)/len(df)*100:.1f}%):")
print(f"  Con recurrencia: {(test_df['recurrencia'] == 1).sum()} ({(test_df['recurrencia'] == 1).sum()/len(test_df)*100:.1f}%)")
print(f"  Sin recurrencia: {(test_df['recurrencia'] == 0).sum()} ({(test_df['recurrencia'] == 0).sum()/len(test_df)*100:.1f}%)")

train_df.to_csv(OUTPUT_TRAIN, index=False, encoding='utf-8')
val_df.to_csv(OUTPUT_VAL, index=False, encoding='utf-8')
test_df.to_csv(OUTPUT_TEST, index=False, encoding='utf-8')

print(f"\nArchivos guardados:")
print(f"  {OUTPUT_TRAIN}")
print(f"  {OUTPUT_VAL}")
print(f"  {OUTPUT_TEST}")


Total filas: 998
Con recurrencia: 235
Sin recurrencia: 763

Train (698 filas, 69.9%):
  Con recurrencia: 164 (23.5%)
  Sin recurrencia: 534 (76.5%)

Validation (150 filas, 15.0%):
  Con recurrencia: 36 (24.0%)
  Sin recurrencia: 114 (76.0%)

Test (150 filas, 15.0%):
  Con recurrencia: 35 (23.3%)
  Sin recurrencia: 115 (76.7%)

Archivos guardados:
  train.csv
  validation.csv
  test.csv
